In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install tensorboardX

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 9.4 MB/s eta 0:00:00


In [ ]:
%cd /content/drive/MyDrive/CSS/PIDNet
!pip install yacs opencv-python-headless easydict pyyaml

/content/drive/MyDrive/CSS/PIDNet


Original

In [ ]:
%cd /content/drive/MyDrive/CSS/PIDNet
!python tools/train.py --cfg configs/cityscapes/pidnet_s_custom_15.yaml

/content/drive/MyDrive/CSS/PIDNet
Seeding with 304
=> creating output/cityscapes/pidnet_s_custom_15
=> creating log/cityscapes/pidnet_small/pidnet_s_custom_15_2025-11-13-19-05
Namespace(cfg='configs/cityscapes/pidnet_s_custom_15.yaml', seed=304, opts=[])
AUTO_RESUME: False
CUDNN:
  BENCHMARK: True
  DETERMINISTIC: False
  ENABLED: True
DATASET:
  DATASET: cityscapes
  EXTRA_TRAIN_SET: 
  NUM_CLASSES: 15
  ROOT: /content/drive/MyDrive/CSS/PIDNet/data/
  TEST_SET: list/cityscapes/val.lst
  TRAIN_SET: list/cityscapes/train.lst
GPUS: (0,)
LOG_DIR: log
LOSS:
  BALANCE_WEIGHTS: [0.4, 1.0]
  CLASS_BALANCE: False
  OHEMKEEP: 131072
  OHEMTHRES: 0.9
  SB_WEIGHTS: 1.0
  USE_OHEM: True
MODEL:
  ALIGN_CORNERS: True
  NAME: pidnet_small
  NUM_OUTPUTS: 2
  PRETRAINED: /content/drive/MyDrive/CSS/PIDNet/pretrained_models/imagenet/PIDNet_S_ImageNet.pth.tar
OUTPUT_DIR: output
PIN_MEMORY: True
PRINT_FREQ: 10
TEST:
  BASE_SIZE: 2048
  BATCH_SIZE_PER_GPU: 4
  FLIP_TEST: False
  IMAGE_SIZE: [2048, 1024]
  M

Updated mIoU Calculation

In [ ]:
%cd /content/drive/MyDrive/CSS/PIDNet
!python tools/train.py --cfg configs/cityscapes/pidnet_s_custom_15.yaml

流式输出内容被截断，只能显示最后 5000 行内容。
Epoch: [48/100] Iter:[40/743], Time: 0.23, lr: [0.005546233319534256], Loss: 1.478736, Acc:0.562310, Semantic loss: 0.639078, BCE loss: 0.406318, SB loss: 0.433341
Epoch: [48/100] Iter:[50/743], Time: 0.23, lr: [0.005544940005548718], Loss: 1.520082, Acc:0.553607, Semantic loss: 0.667401, BCE loss: 0.397322, SB loss: 0.455359
Epoch: [48/100] Iter:[60/743], Time: 0.24, lr: [0.005543646658045043], Loss: 1.557254, Acc:0.560003, Semantic loss: 0.692960, BCE loss: 0.403677, SB loss: 0.460617
Epoch: [48/100] Iter:[70/743], Time: 0.23, lr: [0.005542353277013677], Loss: 1.576246, Acc:0.558345, Semantic loss: 0.701333, BCE loss: 0.405391, SB loss: 0.469522
Epoch: [48/100] Iter:[80/743], Time: 0.23, lr: [0.005541059862445053], Loss: 1.575425, Acc:0.556620, Semantic loss: 0.698693, BCE loss: 0.402057, SB loss: 0.474675
Epoch: [48/100] Iter:[90/743], Time: 0.23, lr: [0.005539766414329604], Loss: 1.578593, Acc:0.558230, Semantic loss: 0.697754, BCE loss: 0.403143, SB loss

Corrected 15 classes weight

In [ ]:
%cd /content/drive/MyDrive/CSS/PIDNet
!python tools/train.py --cfg configs/cityscapes/pidnet_s_custom_15.yaml

/content/drive/MyDrive/CSS/PIDNet
Seeding with 304
=> creating output/cityscapes/pidnet_s_custom_15
=> creating log/cityscapes/pidnet_small/pidnet_s_custom_15_2025-11-14-01-12
Namespace(cfg='configs/cityscapes/pidnet_s_custom_15.yaml', seed=304, opts=[])
AUTO_RESUME: False
CUDNN:
  BENCHMARK: True
  DETERMINISTIC: False
  ENABLED: True
DATASET:
  DATASET: cityscapes
  EXTRA_TRAIN_SET: 
  NUM_CLASSES: 15
  ROOT: /content/drive/MyDrive/CSS/PIDNet/data/
  TEST_SET: list/cityscapes/val.lst
  TRAIN_SET: list/cityscapes/train.lst
GPUS: (0,)
LOG_DIR: log
LOSS:
  BALANCE_WEIGHTS: [0.4, 1.0]
  CLASS_BALANCE: False
  OHEMKEEP: 131072
  OHEMTHRES: 0.9
  SB_WEIGHTS: 1.0
  USE_OHEM: True
MODEL:
  ALIGN_CORNERS: True
  NAME: pidnet_small
  NUM_OUTPUTS: 2
  PRETRAINED: /content/drive/MyDrive/CSS/PIDNet/pretrained_models/imagenet/PIDNet_S_ImageNet.pth.tar
OUTPUT_DIR: output
PIN_MEMORY: True
PRINT_FREQ: 10
TEST:
  BASE_SIZE: 2048
  BATCH_SIZE_PER_GPU: 6
  FLIP_TEST: False
  IMAGE_SIZE: [2048, 1024]
  M

In [ ]:
import shutil, os, glob
from pathlib import Path
from PIL import Image
import numpy as np

In [ ]:
orig = Path("/content/drive/MyDrive/CSS/Later/gtFine")
dest = Path("/content/drive/MyDrive/CSS/PIDNet/data/cityscapes/gtFine")

# remove old folder if it exists
if dest.exists():
    shutil.rmtree(dest)

# copy the entire gtFine tree into gtFine15
# this keeps labelIds.png, color.png, polygons.json, instanceIds.png, etc.
shutil.copytree(orig, dest)

# 2) Remap only the labelTrainIds files inside gtFine15
paths = glob.glob(str(dest / "**" / "*_labelTrainIds.png"), recursive=True)
print("labelTrainIds to remap:", len(paths))
changed_pixels = 0
for p in paths:
    arr = np.array(Image.open(p))
    mask = (arr >= 15) & (arr != 255)  # keep 0..14, turn 15..18 into 255
    changed_pixels += int(mask.sum())
    arr[mask] = 255
    Image.fromarray(arr.astype(np.uint8)).save(p)
print("Total pixels set to 255:", changed_pixels)

# 3) Quick sanity checks
print("val labelTrainIds count:",
      len(glob.glob(str(dest / "val" / "**" / "*_labelTrainIds.png"), recursive=True)))
print("example file exists:",
      (dest / "train" / "aachen").exists())

labelTrainIds to remap: 5000
Total pixels set to 255: 65981454
val labelTrainIds count: 500
example file exists: True


**PIDNet-S on original 15 classes**

In [ ]:
%cd /content/drive/MyDrive/CSS/PIDNet
!python tools/train.py --cfg configs/cityscapes/pidnet_s_custom_15.yaml

流式输出内容被截断，只能显示最后 5000 行内容。
20
30
40
50
60
70
80
--- Validation Output 0 ---
  Original mIoU: 0.535359
  Metric 1 mIoU: 0.535359 (PixAcc: 0.913819)
  Metric 2 mIoU: 0.535359 (PixAcc: 0.913819)
  Metric 3 mIoU: 0.535359 (PixAcc: 0.913819)
  Original IoU Array: [0.9436 0.6634 0.8354 0.1531 0.2582 0.4293 0.4915 0.5856 0.8849 0.4760
 0.8683 0.5971 0.0024 0.8382 0.0034]
---------------------------------
--- Validation Output 1 ---
  Original mIoU: 0.739067
  Metric 1 mIoU: 0.739067 (PixAcc: 0.952368)
  Metric 2 mIoU: 0.739067 (PixAcc: 0.952368)
  Metric 3 mIoU: 0.739067 (PixAcc: 0.952368)
  Original IoU Array: [0.9755 0.8141 0.9097 0.5568 0.5538 0.5681 0.5737 0.7265 0.9126 0.6231
 0.9266 0.7373 0.5930 0.9375 0.6777]
---------------------------------
=> saving checkpoint to output/cityscapes/pidnet_s_custom_15checkpoint.pth.tar
Loss: 1.900, MeanIU:  0.7391, Best_mIoU:  0.7564
[0.97553996 0.81410814 0.90968314 0.5568178  0.55378518 0.56805623
 0.57367533 0.72653529 0.91255269 0.62314053 0.9266

In [ ]:
!python tools/visualize.py \
  --cfg configs/cityscapes/pidnet_s_custom_15.yaml \
  --model-path output/cityscapes/pidnet_s_custom_15/best.pt \
  --num-samples 10 \
  --output-dir validation_results

=> Creating model: pidnet_small
=> Loading weights from: output/cityscapes/pidnet_s_custom_15/best.pt
=> Loaded 479 keys successfully.
=> Visualizing 10 samples...
  2% 10/500 [00:16<13:05,  1.60s/it]

=> Done! Check the 'validation_results' folder for your results.


**PIDNet-M on original 15 classes**

In [ ]:
import os
print(os.cpu_count())

12


In [ ]:
%cd /content/drive/MyDrive/CSS/PIDNet
!python tools/train.py --cfg configs/cityscapes/pidnet_m_custom_15.yaml

流式输出内容被截断，只能显示最后 5000 行内容。
20
30
40
50
60
70
80
--- Validation Output 0 ---
  Original mIoU: 0.576987
  Metric 1 mIoU: 0.576986 (PixAcc: 0.923883)
  Metric 2 mIoU: 0.576986 (PixAcc: 0.923883)
  Metric 3 mIoU: 0.576986 (PixAcc: 0.923883)
  Original IoU Array: [0.9465 0.6825 0.8547 0.1821 0.3015 0.4699 0.5600 0.6645 0.8954 0.4860
 0.8996 0.6704 0.1091 0.8699 0.0626]
---------------------------------
--- Validation Output 1 ---
  Original mIoU: 0.770127
  Metric 1 mIoU: 0.770127 (PixAcc: 0.957397)
  Metric 2 mIoU: 0.770127 (PixAcc: 0.957396)
  Metric 3 mIoU: 0.770127 (PixAcc: 0.957396)
  Original IoU Array: [0.9784 0.8276 0.9147 0.4472 0.6125 0.5954 0.7102 0.7781 0.9177 0.6215
 0.9404 0.8169 0.7079 0.9471 0.7362]
---------------------------------
=> saving checkpoint to output/cityscapes/pidnet_m_custom_15checkpoint.pth.tar
Loss: 1.936, MeanIU:  0.7701, Best_mIoU:  0.7744
[0.97838193 0.82760219 0.91467965 0.44716743 0.61252452 0.59541068
 0.71020768 0.77814872 0.91771348 0.62150433 0.9404

**PIDNet-L on original 15 classes**

In [ ]:
%cd /content/drive/MyDrive/CSS/PIDNet
!python tools/train.py --cfg configs/cityscapes/pidnet_l_custom_15.yaml

Streaming output truncated to the last 5000 lines.
20
30
40
50
60
70
80
--- Validation Output 0 ---
  Original mIoU: 0.591049
  Metric 1 mIoU: 0.591049 (PixAcc: 0.927501)
  Metric 2 mIoU: 0.591049 (PixAcc: 0.927501)
  Metric 3 mIoU: 0.591049 (PixAcc: 0.927501)
  Original IoU Array: [0.9540 0.7099 0.8588 0.1797 0.3317 0.4866 0.5515 0.6606 0.8960 0.4711
 0.8994 0.6623 0.1913 0.8786 0.1341]
---------------------------------
--- Validation Output 1 ---
  Original mIoU: 0.759847
  Metric 1 mIoU: 0.759847 (PixAcc: 0.956539)
  Metric 2 mIoU: 0.759847 (PixAcc: 0.956539)
  Metric 3 mIoU: 0.759847 (PixAcc: 0.956539)
  Original IoU Array: [0.9801 0.8343 0.9105 0.4223 0.5114 0.6123 0.7064 0.7716 0.9169 0.6030
 0.9351 0.8145 0.7213 0.9471 0.7112]
---------------------------------
=> saving checkpoint to output/cityscapes/pidnet_l_custom_15checkpoint.pth.tar
Loss: 1.939, MeanIU:  0.7598, Best_mIoU:  0.7777
[0.98010376 0.8343338  0.91045121 0.42225768 0.5114089  0.61228439
 0.70635806 0.77156833 0.91